# Import Library

In [ ]:
# data download from kaggle
import kagglehub
# data manipulation
import numpy as np
import pandas as pd
# extracting url
import re
# for KNN fill null
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# Load Dataset

In [ ]:
# dataset is downloaded from https://www.kaggle.com/datasets/andrewmvd/okcupid-profiles
path = kagglehub.dataset_download("andrewmvd/okcupid-profiles")

In [ ]:
# load csv from the just downloaded dataset
df = pd.read_csv(path + "/okcupid_profiles.csv")
# load csv if you have already downloaded the dataset
# df = pd.read_csv("okcupid_profiles.csv")

# Simple Investigation

In [ ]:
# Check top 5 rows of dataset
df.head()

In [ ]:
# check column name, data type
df.info()

In [ ]:
# check numerical data 
df.describe()

In [ ]:
# check null value count
df.isnull().sum()

# Data Cleanse

## Transform Data

In [ ]:
def process_sex_height(df):
    """"
    This function will convert
    - 'm' to male and 'f' to female
    - height from inches to cm
    """
    # make a copy of the dataframe
    df = df.copy()
    # convert 'm' to 'male' and 'f' to 'female'
    df['sex'] = df['sex'].map({'m': 'male', 'f': 'female'})
    # convert height from inches to cm
    df['height'] = round(df['height'] * 2.54)
    return df

## Fill Null by Meaningful text

In [ ]:
def fill_missing_values(df):
    """
    This function will fill missing values with meaningful text, combine essays into one column and drop columns that are not needed

    """
    # make a copy of the dataframe
    df = df.copy()
    # fill missing values with meaningful text
    df['diet'] = df['diet'].fillna("anything")
    df['drinks'] = df['drinks'].fillna("not at all")
    df['drugs'] = df['drugs'].fillna("never")
    df['height'] = df['height'].fillna(df['height'].median())
    df['job'] = df['job'].fillna('other')
    df['offspring'] = df['offspring'].fillna("no kids and neutral to kids")
    df['pets'] = df['pets'].fillna("no pets and neutral to pets")
    df['religion'] = df['religion'].fillna("irreligion")
    df['sign'] = df['sign'].str.replace("&rsquo;", "'").fillna("unknown zodiac sign")
    df['smokes'] = df['smokes'].fillna("no")
    df['speaks'] = df['speaks'].fillna("english")
    # filling missing values for essays, combine essays into one column and remove excess ","
    df['essay_all'] = df.loc[:,"essay0":"essay9"].fillna("").apply(lambda x: ','.join(x.astype(str)),axis=1).str.strip(",").fillna("")
    # drop columns that are not needed
    columns_to_drop = ['essay0', 'essay1', 'essay2', 'essay3', 'essay4', 'essay5', 'essay6','essay7', 'essay8', 'essay9', 'last_online']
    df = df.drop(columns=columns_to_drop)
    
    return df

## Fill Value by KNN

In [ ]:
def fill_missing_values_knn(df, col_fill, list_col_ref, list_col_num):
    """
    This function will fill missing values using KNN imputer
    """

    # Copy dataframe to avoid modifying original dataframe
    df = df.copy()

    # Create a label encoder object
    label_encoder = LabelEncoder()

    # drop rows with missing values in the column to be imputed
    df_non_null = df[col_fill].dropna()
    # Fit and transform the column to be imputed
    df.loc[df_non_null.index, col_fill +'_encoded'] = label_encoder.fit_transform(df_non_null)

    # Convert the column to float
    df[col_fill + '_encoded'] = df[col_fill + '_encoded'].astype(float)
    # Fill missing values with NaN
    df.loc[df[col_fill].isnull(), col_fill + '_encoded'] = np.nan

    # Print the mapping of the label encoder
    print("Label Encoding Mapping: ", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

    # Convert categorical variables into numerical format
    for col in list_col_ref:
        df[col + '_encoded'] = LabelEncoder().fit_transform(df[col].astype(str))  # Encode as numbers

    # List of features to be used for KNN including numerical columns, encoded categorical columns and encoded column to be imputed
    knn_features = list_col_num + [col + '_encoded' for col in list_col_ref] + [col_fill + '_encoded']

    # Ensure only features to be used for KNN are selected
    df_knn = df[knn_features]

    # Initialize KNN Imputer with 5 neighbors
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    # Fit and transform the KNN imputer
    df_knn_imputed = imputer.fit_transform(df_knn)

    # Replace original dataframe values with imputed values
    df[knn_features] = df_knn_imputed

    # Round to nearest integer before inverse transforming
    df[col_fill + '_encoded'] = df[col_fill + '_encoded'].round().astype(int)

    # Inverse transform the encoded column to get the original values
    df[col_fill + '_imputed'] = label_encoder.inverse_transform(df[col_fill + '_encoded'])

    # Replace the original column with the imputed values
    df[col_fill] = df[col_fill + '_imputed']

    # Drop columns that are no longer needed
    df = df.drop(columns=[col_fill + '_encoded', col_fill + '_imputed'] + [col + '_encoded' for col in list_col_ref])

    return df

def fill_missing_values_knn_multi_col(df, col_fillna):
    """
    This function will fill missing values using KNN imputer for multiple columns
    """
    # Copy dataframe to avoid modifying original dataframe
    df = df.copy()
    # fill missing values for each key in the dictionary
    for col_fill, list_col_ref in col_fillna.items():
        df = fill_missing_values_knn(df, col_fill, list_col_ref[0], list_col_ref[1])
    return df

## Extract and Remove URL

In [ ]:
def extract_urls(text):
    """
    This function will extract URLs from text for one row
    """
    # Check if text is a string
    if not isinstance(text, str):
        return []
    # extract URLs
    url_pattern = r'\b(?=https?:\/\/|www\.[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6})\b([-a-zA-Z0-9(@:%_\+.~#?&//=]*)'
    # Extract matches
    matches = re.findall(url_pattern, text)
    return matches

def remove_url(row):
    """"
    This function will remove URL from text for one row
    """
    # Get essay text
    essay = row['essay_all']
    # Remove URL from text
    for url in row['url']:
        essay = essay.replace(url, '')
    return essay

def remove_url_col(df):
    """
    This function will remove URL from text for each row in the dataframe
    """
    # Copy dataframe to avoid modifying original dataframe
    df = df.copy()
    # Extract URLs for each row
    df['url'] = df['essay_all'].apply(extract_urls)
    # Remove URLs for each row
    df['essay_all'] = df.apply(remove_url, axis=1)
    return df

## Combine Cleanse Data Process

In [ ]:
def data_cleaning(df, col_fillna):
    """
    This function will clean the dataset by:
    - Converting
        - 'm' to 'male'
        - 'f' to 'female'
        - height from inches to cm 
    - Filling missing values with meaningful text
    - Filling missing values using KNN imputer
    - Removing URL from text
    """
    # Copy dataframe to avoid modifying original dataframe
    df = df.copy()
    df = process_sex_height(df)
    df = fill_missing_values(df)
    df = fill_missing_values_knn_multi_col(df, col_fillna)
    df = remove_url_col(df)
    # Drop columns that are not needed
    df = df.drop(columns=['income','url'])
    return df

# Perform Data Cleansing

In [ ]:
# Dictionary of columns to fillna as key, list of columns to be used as reference and numerical columns as value
col_fillna = {'body_type': [['sex', 'drinks', 'diet', 'drugs'], ['age', 'height']],
              'education': [['job', 'location'], ['age', 'income']],
              'ethnicity': [['religion', 'location'], ['age']]}

In [ ]:
# Call the data_cleaning function
df_cleaned = data_cleaning(df, col_fillna)

In [ ]:
# Check top 5 rows of cleaned dataset
df_cleaned.head()

# Save Cleansed Dataset as .csv

In [ ]:
# save cleaned dataset as csv
df_cleaned.to_csv("okcupid_profiles_cleaned.csv", index=False)

In [ ]:
# check if the cleaned dataset is saved correctly
pd.read_csv("okcupid_profiles_cleaned.csv").fillna("").equals(df_cleaned)